# Multi-Agent Collaboration

Topologies, handoffs, orchestrator-worker, and when multi-agent hurts.

This notebook is an original tutorial (rewritten for depth, 2026 practice).
Deep-dive companion: notebook 28 (production agent harness).

## Learning Objectives

- Compare the main topologies — supervisor/orchestrator-worker, network, hierarchical, handoff — and their tradeoffs.
- Explain handoffs (transfer of control) vs delegation (subagent as tool call) precisely.
- State the token economics (~15x) and the context-isolation rationale for subagents.
- Argue when NOT to use multi-agent — the most senior answer in the topic.

## Mental Model

Multi-agent is a way to buy two things a single context can't provide:
**more total thinking tokens** (each subagent has its own window) and
**isolation** (exploration garbage in one agent doesn't pollute another).
Everything else — "specialist roles", "teams of agents" — is framing.
The costs are real: ~15x tokens vs chat, coordination failures, and
duplicated work. So the design question is never "which agents?" but
"is this task parallelizable and valuable enough to pay 15x?"

Communication rule that makes it all work: agents exchange **structured,
compressed results — never shared raw context.**

## Important Concepts

- **Orchestrator-worker (supervisor)**: a lead agent plans, spawns workers with explicit mandates, synthesizes results. The dominant production topology (Anthropic's research system: lead plans, 3-5 parallel subagents, 90.2% improvement over single-agent, ~80% of variance explained by token spend).
- **Handoff**: *transfer of control* — agent A stops, agent B takes over the conversation with the user (OpenAI Agents SDK pattern; triage → specialist). The user talks to one agent at a time.
- **Delegation (subagent-as-tool)**: agent A calls agent B like a tool and consumes its result; A stays in charge. Most "multi-agent" systems are this.
- **Network/debate**: peer agents exchange messages (critique, vote). Expensive; used for judge panels and adversarial verification more than open collaboration.
- **Hierarchical**: supervisors of supervisors; adds coordination overhead fast.
- **Mandate**: the spawn prompt for a worker — objective, output format, tool guidance, effort budget. Vague mandates were Anthropic's #1 early failure (duplicate work, over-spawning).

## Need-To-Know Coverage Checklist

- [x] Topologies with tradeoffs: orchestrator-worker, handoff, network/debate, hierarchical.
- [x] Handoff vs delegation distinction.
- [x] Token economics: chat ~1x, single agent ~4x, multi-agent ~15x.
- [x] Context isolation as the point; structured results as the interface.
- [x] Mandate design; effort scaling (simple query = 1 agent, 3-10 calls).
- [x] When multi-agent hurts: sequential tightly-coupled work, low task value.
- [x] Fan-out/fan-in mechanics and result merging (reducers — notebook 28).

## Deep Study Notes

### Orchestrator-worker, the production default

The lead agent decomposes the task and spawns workers *in parallel*, each
with an explicit mandate. Workers explore in their own context windows and
return compressed findings; the lead synthesizes. Two design rules from
production experience:

1. **Detailed mandates prevent duplicate work.** "Research competitor
   pricing" spawned three times produces three overlapping reports. Mandates
   must partition the space: objective, sources to use, output schema,
   effort budget.
2. **Scale effort to complexity.** Simple fact-check: one agent, a few tool
   calls. Broad research: 3-5 agents. Spawning ten agents for everything is
   how you get 15x cost with 1x quality.

### Handoff vs delegation (a precise distinction interviewers reward)

- **Delegation**: the caller keeps the conversation; the subagent is
  invisible to the user; its final message is consumed as data. Failure mode:
  the caller must know how to *use* the result.
- **Handoff**: control transfers; the new agent owns the user conversation
  and its history slice. Failure mode: context loss at the boundary — what
  does the specialist know about what came before? Production handoffs pass
  a structured summary, not the raw transcript.

### When multi-agent hurts

- Tightly-coupled sequential work (most single-feature coding): agents need
  each other's intermediate state, so you pay coordination cost without
  parallel speedup.
- Low-value tasks: 15x token spend needs 15x-worthy output.
- Shared-context designs: two agents editing one conversation compound each
  other's errors and blow the window — the anti-pattern the
  structured-results rule exists to prevent.

The senior answer to "should we make this multi-agent?" is usually
"show me the parallelizable decomposition first."''

## Common Failure Modes

- Vague mandates → duplicate and missed work across workers.
- Raw-context sharing → token blowup, cross-contaminated reasoning.
- Handoff without a structured summary → specialist re-asks everything the user already said.
- Multi-agent applied to sequential work → slower and 15x more expensive than one agent.
- No fan-in reducer → parallel results clobber or get pasted unsynthesized.
- Workers with unbounded budgets → one stuck worker stalls the whole fan-out.

## Implementation Notes

- Write the mandate template once: objective / scope boundaries / output schema / tool allowlist / budget.
- Workers return structured results (schema-validated); the orchestrator never reads worker transcripts.
- Time-box workers; treat a timeout as a partial result, not a failure of the run.
- Log the plan and every mandate — post-hoc "why did it spawn 9 agents?" must be answerable.

In [ ]:
"""Orchestrator-worker vs handoff, both in ~40 lines.

Fake agents demonstrate the two control-flow shapes and the
structured-results rule.
"""

# --- delegation: orchestrator-worker fan-out/fan-in ------------------------
def worker(mandate):
    """A worker returns a STRUCTURED result — never its 'transcript'."""
    findings = {
        "pricing":  {"finding": "3 tiers, $29-299", "confidence": "high", "sources": 4},
        "reviews":  {"finding": "support complaints trending up", "confidence": "medium", "sources": 9},
        "hiring":   {"finding": "5 open ML roles", "confidence": "high", "sources": 2},
    }
    return {"mandate": mandate["objective"], **findings[mandate["topic"]]}


def orchestrator(task):
    plan = [  # explicit, partitioned mandates — no overlap
        {"topic": "pricing", "objective": "map competitor pricing tiers", "budget_calls": 5},
        {"topic": "reviews", "objective": "summarize customer sentiment", "budget_calls": 8},
        {"topic": "hiring",  "objective": "infer strategy from job posts", "budget_calls": 3},
    ]
    results = [worker(m) for m in plan]        # parallel in production
    high_conf = [r for r in results if r["confidence"] == "high"]
    return {"task": task, "findings": results,
            "synthesis": f"{len(high_conf)}/{len(results)} findings high-confidence"}


report = orchestrator("competitive analysis of AcmeCo")
for f in report["findings"]:
    print(f"  [{f['confidence']:>6}] {f['mandate']}: {f['finding']}")
print(report["synthesis"])

# --- handoff: transfer of control with a structured summary ----------------
def triage_agent(user_msg):
    if "refund" in user_msg:
        summary = {"intent": "refund", "order": "ORD-7", "already_verified": True}
        return {"handoff_to": "billing", "context_summary": summary}
    return {"reply": "How can I help?"}


def billing_agent(context_summary, user_msg):
    v = "verified" if context_summary["already_verified"] else "needs verification"
    return f"Billing here - processing refund for {context_summary['order']} ({v})."


msg = "I want a refund for order ORD-7"
t = triage_agent(msg)
print("\nhandoff:", billing_agent(t["context_summary"], msg))
# Note what crossed the boundary: a compact summary, NOT the transcript.


## Practice

1. Add a fourth worker whose topic overlaps `reviews`. Show the duplicate
   work in the output, then fix it by tightening both mandates.
2. Give one worker `confidence: "low"` and make the orchestrator re-spawn it
   once with a narrower mandate before synthesizing.
3. Break the handoff: pass the raw message instead of `context_summary` and
   explain what the billing agent now has to re-ask.
4. A PM proposes "a team of 12 specialist agents" for code review. Using the
   token economics and the parallelizability test, argue for the right number.

## Design Checklist

- [ ] Task decomposition is parallelizable before any multi-agent design starts.
- [ ] Mandates: objective, scope boundary, output schema, tool allowlist, budget.
- [ ] Workers return schema-validated structured results; no transcript sharing.
- [ ] Fan-in has an explicit reducer/synthesis step.
- [ ] Handoffs pass structured context summaries.
- [ ] Effort scaled to task complexity; worker count justified.
- [ ] Every worker time-boxed; partial results handled.